In [ ]:
# 1. Spark session and imports
#
# On EMR notebooks, Spark is usually already available as a variable named `spark`.
# This fallback makes the notebook runnable in a plain PySpark environment too.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("foodaps-practical-pyspark-learning")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark version:", spark.version)
print("Application ID:", sc.applicationId)

In [ ]:
# 2. S3 paths
#
# Change only these values for your environment.
# All reads and writes in this notebook happen through S3 paths.

S3_BUCKET = "your-bucket-name"
BASE_PREFIX = "spark-learning/foodaps"

RAW_BASE_PATH = f"s3://{S3_BUCKET}/{BASE_PREFIX}/raw"
OUTPUT_BASE_PATH = f"s3://{S3_BUCKET}/{BASE_PREFIX}/output"

print("Raw path:", RAW_BASE_PATH)
print("Output path:", OUTPUT_BASE_PATH)

In [ ]:
# DataFrame without custom schema.
# Spark infers column types from the Python values.

dark_store_rows = [
    ("DS_001", "Bengaluru", "Koramangala", 12, True),
    ("DS_002", "Bengaluru", "Indiranagar", 15, True),
    ("DS_003", "Mumbai", "Powai", 18, True),
    ("DS_004", "Delhi NCR", "Gurugram", 20, False),
]

# This will create dataframe with random column names
# dark_stores_inferred_df = spark.createDataFrame(
#     dark_store_rows
# )

dark_stores_inferred_df = spark.createDataFrame(
    dark_store_rows,
    ["store_id", "city", "zone", "target_delivery_minutes", "is_active"]
)

dark_stores_inferred_df.printSchema()
dark_stores_inferred_df.show(truncate=False)

In [ ]:
# DataFrame with custom schema.
# This is the production-friendly way to create small reference/config DataFrames.
from pyspark.sql import types as T

sla_schema = T.StructType([
    T.StructField("order_channel", T.StringType(), nullable=False),
    T.StructField("priority_level", T.StringType(), nullable=False),
    T.StructField("target_minutes", T.IntegerType(), nullable=False),
    T.StructField("free_delivery_threshold", T.DoubleType(), nullable=True),
])

sla_rows = [
    ("quick_commerce", "high", 10, 199.0),
    ("quick_commerce", "standard", 20, 99.0),
    ("scheduled_delivery", "standard", 120, 299.0),
]

sla_rules_df = spark.createDataFrame(sla_rows, schema=sla_schema)

sla_rules_df.printSchema()
sla_rules_df.show(truncate=False)

In [ ]:
# Core FoodAPS files used in this notebook.
#
# faps_fahevent_puf.csv: food-at-home acquisition events, similar to grocery purchase events.
# faps_fahitem_puf.csv: items inside food-at-home events.
# faps_fafhevent_puf.csv: food-away-from-home events, useful for contrast.
# faps_fafhitem_puf.csv: items inside food-away-from-home events.
# faps_household_puf.csv: household/customer dimension.

foodaps_files = {
    "fah_events": f"{RAW_BASE_PATH}/faps_fahevent_puf.csv",
    "fah_items": f"{RAW_BASE_PATH}/faps_fahitem_puf.csv",
    "fafh_events": f"{RAW_BASE_PATH}/faps_fafhevent_puf.csv",
    "fafh_items": f"{RAW_BASE_PATH}/faps_fafhitem_puf.csv",
    "households": f"{RAW_BASE_PATH}/faps_household_puf.csv",
}

# CSV read options:
#
# header=true:
#   First row contains column names.
#
# inferSchema=true:
#   Spark scans data to infer types. Good for exploration, slower and less stable
#   than explicit schema in production.
#
#
# multiLine=false:
#   FoodAPS files are normal row-per-line CSVs. Keep multiLine disabled here.
#   If multiLine=true is enabled on this dataset, Spark uses MultiLineCSVDataSource
#   during inferSchema and may throw a Univocity ArrayIndexOutOfBoundsException
#   because a problematic quote can make the parser treat many rows as one record.
#   Enable multiLine only when one logical CSV row actually spans multiple lines.

csv_common_read_options = {
    "multiLine": "false",
    "quote": '"',
    "escape": '"',
}

fah_events_raw =  spark.read \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .options(**csv_common_read_options) \
                    .csv(foodaps_files["fah_events"])

fah_events_raw.printSchema()
fah_events_raw.show(truncate=False)

fah_items_raw =  spark.read \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .options(**csv_common_read_options) \
                    .csv(foodaps_files["fah_items"])

fah_items_raw.printSchema()
fah_items_raw.show(truncate=False)

households_raw =  spark.read \
                    .option("header", "true") \
                    .option("inferSchema", "true") \
                    .options(**csv_common_read_options) \
                    .csv(foodaps_files["households"])

households_raw.printSchema()
households_raw.show(truncate=False)

In [ ]:
# Quick column check.

print("FAH event columns:")
print(fah_events_raw.columns)

print("\nFAH item columns:")
print(fah_items_raw.columns)

print("\nHousehold columns:")
print(households_raw.columns)

In [ ]:
# Select and withColumn
# `select(...alias(...))` is better when you also cast, derive, or normalize.

fah_events = (
    fah_events_raw
    .select(
        F.col("hhnum").cast("long").alias("household_id"),
        F.col("eventid").cast("long").alias("event_id"),
        F.col("daynum").cast("int").alias("survey_day"),
        F.col("placeid").cast("long").alias("place_id"),
        F.col("placecateg").cast("string").alias("place_category"),
        F.col("placetype").cast("string").alias("place_type"),
        F.col("totalpaid").cast("double").alias("event_total_paid")
    )
    .withColumn("event_source", F.lit("food_at_home"))
)

fah_events.show(5, truncate=False)